# Transform — boceto del flujo de transformación

Borrador de la etapa **Transform** del pipeline. Acá se prueba la lógica contra la data
disponible; una vez validada, se traslada tal cual a `src/transform.py`.

El diseño sale de las reglas documentadas en [`eda.ipynb`](eda.ipynb): **R2–R12**
(calidad de parseo) y **D0–D7** (distribución y outliers).

## Dos stages de salida

Con el barrido por tipo el dataset proyectado ronda las **60 000 filas**, y con ese
volumen conviene que al análisis llegue sólo lo confiable. El flujo termina en dos
salidas:

- **`anuncios`** — las filas que pasan todas las reglas bloqueantes.
- **`cuarentena`** — las rechazadas, cada una con su `motivo_rechazo`.

No es un descarte: es una separación. La cuarentena queda en disco, auditable, y si una
regla resulta demasiado estricta se reprocesa desde ahí.

**Ojo con qué bloquea y qué no.** Las banderas `_es_tope` marcan censura del origen, no
un defecto — filtrarlas se llevaría el 26 % de las filas y con ellas todo el segmento
alto del mercado. Pasan al stage limpio con su bandera puesta. El detalle está en la
sección 8.

## Requisito de compatibilidad

El extract cambió: ahora hace seis barridos (3 tipos × 2 operaciones) y emite
`id_inmueble`, `tipo_inmueble`, `operacion` y `fecha_extraccion` como columnas.

Este flujo tiene que funcionar con **los dos esquemas**, para poder validarse contra la
data vieja y seguir sirviendo con la nueva. La normalización de esquema es el primer
paso, y todo lo demás trabaja sobre el esquema normalizado.

| | esquema viejo | esquema nuevo |
|---|---|---|
| identidad | derivar de `url` | `id_inmueble` |
| **tipo** | **slug de la `url`** | **slug de la `url`** (la etiqueta del barrido queda como `tipo_feed`) |
| operación | inferir del nombre del archivo | `operacion` |
| fecha | no existe | `fecha_extraccion` |

### Por qué el tipo sale del slug y no del barrido

La etiqueta del barrido parecía autoritativa y no lo es. Falla por dos vías distintas:

1. **Es por lote, no por anuncio.** Un anuncio que aparece en dos barridos se queda con
   la etiqueta del primero que corrió, no con la suya.
2. **El filtro de la fuente se cae en las páginas profundas.** El barrido de
   apartaestudios en arriendo devolvió oficinas, bodegas y locales comerciales.

El slug de la URL (`/inmueble/arriendo-apartaestudio-bogota-.../21601-M6844386`) es **por
anuncio** y no tiene ninguno de los dos problemas. Es la fuente independiente que faltaba:
sin ella, validar `tipo_inmueble` era comparar nuestra propia etiqueta contra sí misma.

## Orden de aplicación

El orden importa y no es negociable (ver D3 en el EDA):

```
1. Normalizar esquema      →  un solo contrato de columnas + tipo desde el slug (R3)
2. Operación y duales      →  R4 + D0: separa precio_venta / precio_arriendo
3. Saneamiento de precio   →  R5, R6
4. Saneamiento de área     →  R7 + D4 (area_posible_lote)
5. Derivar precio_m2       →  sólo sobre lo que sobrevivió a 3 y 4
6. Outliers IQR log        →  D1, D2, D3
7. Consolidar y separar    →  R11 + compuerta de calidad
```

Calcular `precio_m2` antes del paso 2 es el error clásico: se divide un precio de venta
que estaba en el feed de arriendo y el indicador nace roto.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

DIR_RAW = Path("../data/raw")
DIR_PROCESSED = Path("../data/processed")

RESIDENCIAL = ("apartaestudio", "apartamento", "casa")


def vallas_iqr_log(serie, k=1.5):
    """Vallas de Tukey sobre log(x), devueltas en la escala original.

    En log porque estas variables son log-normales: sobre la escala cruda la valla
    inferior da negativa y no detecta ni un outlier bajo (ver D1 en el EDA).
    """
    s = pd.to_numeric(serie, errors="coerce").dropna()
    s = s[s > 0]
    if len(s) < 4:
        return (np.nan, np.nan)
    x = np.log(s)
    q1, q3 = x.quantile(.25), x.quantile(.75)
    iqr = q3 - q1
    return float(np.exp(q1 - k * iqr)), float(np.exp(q3 + k * iqr))

## 1. Normalización de esquema — R2 + R3

Un único `cargar()` que acepta las dos formas del CSV y devuelve siempre las mismas
columnas. Las diferencias se resuelven acá y nada más abajo vuelve a preguntarse de qué
esquema viene el dato.

Detalles que resuelve:

- **`id_inmueble`** (**R2**): último segmento de la ruta de `url`, sin querystring. En el
  esquema viejo hay que derivarlo; en el nuevo ya viene. Es la llave.
- **`tipo_inmueble`** (**R3**): se deriva del **slug de la URL**, en los dos esquemas. Es
  la única fuente por anuncio; ni el título (regex frágil) ni la etiqueta del barrido
  (por lote, y contaminada) sirven. Lo que no cae en los tres tipos residenciales queda
  marcado con `tipo_no_residencial` y va a cuarentena.
- **`tipo_feed`**: la etiqueta del barrido, conservada **sólo para auditar**. La diferencia
  entre `tipo_feed` y `tipo_inmueble` es exactamente la medida del problema.
- **`operacion_feed`**: de qué barrido salió la fila. En el esquema viejo se toma del
  nombre del archivo. Ojo: **no** es lo mismo que la operación declarada en el título —
  esa distinción es justamente lo que destapa los duales.

In [2]:
PATRON_TITULO = re.compile(
    r"(?P<tipo>[A-Za-zÁÉÍÓÚÑáéíóúñ ]+?)\s+en\s+"
    r"(?P<operacion>(?:Venta|Arriendo)(?:\s+y\s+(?:Venta|Arriendo))?),"
    r"(?:\s*(?P<sector>[^,]+),)?\s*(?P<ciudad>[^,\n]+)$",
    flags=re.IGNORECASE,
)

# El tipo real vive en el slug de la URL, siempre en la misma posición:
#   /inmueble/arriendo-apartaestudio-bogota-chapinero-1-habitaciones/21601-M6844386
#             ^^^^^^^^ ^^^^^^^^^^^^^
#             operación   tipo
# Sin re.IGNORECASE a propósito: los slugs son minúscula y con pyarrow instalado las
# regex sin flags las resuelve RE2, que es más rápido.
PATRON_TIPO_SLUG = r"/inmueble/(?:arriendo|venta)-([a-z]+)-"

COLUMNAS_CONTRATO = [
    "id_inmueble", "url", "texto", "tipo_inmueble", "tipo_feed", "operacion_feed",
    "precio_texto", "area_m2", "habitaciones", "banos", "parqueaderos",
    "sector", "ciudad", "fecha_extraccion",
]


def sin_tilde(texto):
    if not isinstance(texto, str):
        return texto
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")


def normalizar_tipo(serie):
    return serie.map(sin_tilde).str.lower().str.strip()


def id_desde_url(serie):
    """Último segmento de la ruta, sin querystring. Formatos: '17548-M6886169' y 'MC6943773'."""
    return serie.str.split("?").str[0].str.rstrip("/").str.rsplit("/", n=1).str[-1]


def tipo_desde_url(serie):
    """R3 — tipo del anuncio según su propio slug. Fuente por anuncio, no por barrido."""
    return serie.str.extract(PATRON_TIPO_SLUG, expand=False).str.strip()


def cargar(ruta, operacion_feed):
    """Lee un CSV del extract (esquema viejo o nuevo) y devuelve el contrato normalizado."""
    df = pd.read_csv(ruta)
    # Tres generaciones del extract conviven en data/raw:
    #   v1  sin tipo: hay que sacarlo del título
    #   v2  `tipo_inmueble` = etiqueta del barrido (la que resultó no confiable)
    #   v3  `tipo_inmueble` = slug del anuncio, `tipo_barrido` = etiqueta del barrido
    esquema = ("v3" if "tipo_barrido" in df.columns
               else "v2" if "tipo_inmueble" in df.columns
               else "v1")

    if "id_inmueble" not in df.columns:
        df["id_inmueble"] = id_desde_url(df["url"])
    if "operacion" in df.columns:
        df["operacion_feed"] = df["operacion"]
    else:
        df["operacion_feed"] = operacion_feed
    if "fecha_extraccion" not in df.columns:
        df["fecha_extraccion"] = pd.NaT

    # Etiqueta de procedencia: se guarda para auditar, NO decide el tipo (ver resolver_tipo).
    # Ojo con el orden: en v3 la etiqueta del barrido ya no vive en `tipo_inmueble` sino en
    # `tipo_barrido`. Leerla del lugar equivocado haría que el acuerdo diera 100 % siempre
    # y el indicador dejaría de detectar nada.
    if esquema == "v3":
        df["tipo_feed"] = normalizar_tipo(df["tipo_barrido"])
    elif esquema == "v2":
        df["tipo_feed"] = normalizar_tipo(df["tipo_inmueble"])
    else:
        df["tipo_feed"] = normalizar_tipo(df["texto"].str.extract(PATRON_TITULO)["tipo"])

    df["tipo_inmueble"] = tipo_desde_url(df["url"])

    faltantes = [c for c in COLUMNAS_CONTRATO if c not in df.columns]
    if faltantes:
        raise ValueError(f"{ruta}: faltan columnas tras normalizar: {faltantes}")

    print(f"  {ruta.name:28} esquema={esquema:3} filas={len(df):6}")
    return df[COLUMNAS_CONTRATO].copy()


def resolver_tipo(df):
    """R3 — manda el slug. Marca lo que no es vivienda para que la compuerta lo saque.

    Un slug ilegible (sin match) también queda marcado: si no se puede afirmar que la
    fila es vivienda, no entra al stage limpio.
    """
    df["tipo_no_residencial"] = ~df["tipo_inmueble"].isin(RESIDENCIAL)

    coincide = df["tipo_inmueble"].eq(df["tipo_feed"])
    print(f"\n  R3 — slug vs. etiqueta del barrido: coinciden {coincide.mean() * 100:.2f} %"
          f"  ({int((~coincide).sum()):,} filas reetiquetadas)")
    if (~coincide).any():
        cruce = (df.loc[~coincide]
                 .groupby(["tipo_feed", "tipo_inmueble"], dropna=False)
                 .size().sort_values(ascending=False).head(10))
        display(cruce.to_frame("filas"))

    fuera = df.loc[df["tipo_no_residencial"], "tipo_inmueble"]
    print(f"  R3 — no residencial (a cuarentena): {len(fuera):,} filas"
          f"  {fuera.value_counts(dropna=False).to_dict()}")
    return df


print("Cargando:")
crudo = pd.concat(
    [cargar(DIR_RAW / f"anuncios_{op}.csv", op) for op in ("arriendo", "venta")],
    ignore_index=True,
)
print(f"\nTotal crudo: {len(crudo):,} filas")
crudo = resolver_tipo(crudo)
print("\ntipo_inmueble final:", crudo["tipo_inmueble"].value_counts(dropna=False).to_dict())
display(crudo.head(3))

Cargando:
  anuncios_arriendo.csv        esquema=v2  filas= 26582


  anuncios_venta.csv           esquema=v2  filas= 24220

Total crudo: 50,802 filas

  R3 — slug vs. etiqueta del barrido: coinciden 89.18 %  (5,495 filas reetiquetadas)


filas
tipo_feed     tipo_inmueble       
apartaestudio apartamento     2708
              oficina          996
              casa             701
              bodega           581
              local            334
              edificio         109
              finca             41
              consultorio       14
              lote              11

  R3 — no residencial (a cuarentena): 2,086 filas  {'oficina': 996, 'bodega': 581, 'local': 334, 'edificio': 109, 'finca': 41, 'consultorio': 14, 'lote': 11}

tipo_inmueble final: {'apartamento': 20119, 'casa': 20106, 'apartaestudio': 8491, 'oficina': 996, 'bodega': 581, 'local': 334, 'edificio': 109, 'finca': 41, 'consultorio': 14, 'lote': 11}


,id_inmueble,url,texto,tipo_inmueble,tipo_feed,operacion_feed,precio_texto,area_m2,habitaciones,banos,parqueaderos,sector,ciudad,fecha_extraccion,tipo_no_residencial
0,9851-M6839735,https://www.metrocuadrado.com/inmueble/arriend...,Destacado El Poblado | Barranquilla $1.769.280...,apartaestudio,apartaestudio,arriendo,$1.769.280,39.0,1.0,1.0,1.0,El Poblado,Barranquilla,2026-09-10T04:25:40+00:00,False
1,22335-M6523328,https://www.metrocuadrado.com/inmueble/arriend...,Destacado CIUDAD DEL RIO | Suroriente | Medell...,apartaestudio,apartaestudio,arriendo,$2.450.777,32.0,1.0,1.0,NaN,CIUDAD DEL RIO,Medellín,2026-09-10T04:25:40+00:00,False
2,21003-M6817169,https://www.metrocuadrado.com/inmueble/arriend...,Destacado Los Monjes | Bogotá D.C. $1.600.000 ...,apartaestudio,apartaestudio,arriendo,$1.600.000,29.0,1.0,1.0,NaN,Los Monjes,Bogotá D.C.,2026-09-10T04:25:40+00:00,False


## 2. Operación declarada y anuncios duales — R4 + D0

La operación del **feed** (de qué barrido salió) no siempre coincide con la operación
declarada en el **título**. Cuando el título dice `"Casa en Venta y Arriendo"`, el
inmueble se ofrece en las dos modalidades y aparece en los dos barridos.

Y acá está el hallazgo de mayor impacto del EDA (**D0**): en el feed de arriendo, los
duales publican el **precio de venta**, no el canon. Los 82 duales del dataset anterior
tenían todos precio > $100 millones mensuales, con una mediana **333 veces** mayor que la
de un arriendo normal.

Por eso el precio no puede quedar en una sola columna. Se separa en `precio_venta` y
`precio_arriendo`, y para un dual el canon queda en `NaN`: no lo conocemos.

In [3]:
UMBRAL_ARRIENDO_IMPOSIBLE = 100_000_000  # ningún canon residencial mensual llega acá


def a_entero(serie):
    """'$1.850.000.000' -> 1850000000. El punto es separador de miles, no decimal."""
    return pd.to_numeric(
        serie.astype(str).str.replace(r"[^\d]", "", regex=True).replace("", np.nan),
        errors="coerce",
    ).astype("Int64")


def resolver_operacion(df):
    partes = df["texto"].str.extract(PATRON_TITULO)
    declarada = partes["operacion"].str.lower().str.strip()

    df["es_dual"] = declarada.str.contains(" y ", na=False)
    df["operacion"] = np.where(df["es_dual"], "ambas", df["operacion_feed"])
    # Si el título no matcheó, la operación del feed sigue siendo válida
    df["titulo_sin_parsear"] = declarada.isna()

    # Sector y ciudad del título sólo si el extract no los trajo (esquema viejo con duales)
    df["sector"] = df["sector"].fillna(partes["sector"].str.strip())
    df["ciudad"] = df["ciudad"].fillna(partes["ciudad"].str.strip())
    return df


def separar_precios(df):
    """D0: en el feed de arriendo, el precio de un dual es de venta, no un canon."""
    precio = a_entero(df["precio_texto"])

    es_venta = df["operacion_feed"].eq("venta")
    df["precio_venta"] = precio.where(es_venta | df["es_dual"])
    df["precio_arriendo"] = precio.where(~es_venta & ~df["es_dual"])
    return df


crudo = resolver_operacion(crudo)
crudo = separar_precios(crudo)

print("Operación resuelta:", crudo["operacion"].value_counts(dropna=False).to_dict())
print("Duales:", int(crudo["es_dual"].sum()), "| títulos sin parsear:", int(crudo["titulo_sin_parsear"].sum()))

# Verificación de D0 contra esta data
duales_arriendo = crudo[crudo["es_dual"] & crudo["operacion_feed"].eq("arriendo")]
normales_arriendo = crudo[~crudo["es_dual"] & crudo["operacion_feed"].eq("arriendo")]
if len(duales_arriendo):
    precio_dual = a_entero(duales_arriendo["precio_texto"])
    print(f"\nD0 — feed de arriendo:")
    print(f"  mediana duales     : ${precio_dual.median():>18,.0f}")
    print(f"  mediana no duales  : ${normales_arriendo['precio_arriendo'].median():>18,.0f}")
    print(f"  duales por encima de ${UMBRAL_ARRIENDO_IMPOSIBLE:,}: "
          f"{int((precio_dual > UMBRAL_ARRIENDO_IMPOSIBLE).sum())} de {len(duales_arriendo)}")

Operación resuelta: {'arriendo': 26325, 'venta': 24179, 'ambas': 298}
Duales: 298 | títulos sin parsear: 0

D0 — feed de arriendo:
  mediana duales     : $     1,800,000,000
  mediana no duales  : $         5,000,000
  duales por encima de $100,000,000: 257 de 257


## 3. Saneamiento de precio — R5 + R6

Dos filtros distintos, en este orden:

**R6 — precios de relleno.** El anunciante escribe una cifra placeholder: dígito repetido
siete veces o más (`$22.222.222.222`, `$999.999.999`) o la secuencia `123456`. Son pocos
casos pero envenenan la cola alta. Se anulan antes de evaluar rangos, porque si no el
rango los da por buenos cuando caen dentro.

**R5 — rangos por operación.** Canon mensual y precio de venta son unidades económicas
distintas; un umbral global no significa nada.

| | mínimo | máximo |
|---|---|---|
| Arriendo (mensual) | $300.000 | $500.000.000 |
| Venta | $20.000.000 | $100.000.000.000 |

In [4]:
RANGO_PRECIO = {
    "arriendo": (300_000, 500_000_000),
    "venta": (20_000_000, 100_000_000_000),
}


# Un solo dígito repetido siete veces o más. Se escribe como alternación explícita y no
# como la retroreferencia (\d)\1{6,}: con pyarrow instalado, pandas resuelve las regex de
# strings con RE2, que no soporta retroreferencias y falla con ArrowInvalid.
PATRON_DIGITO_REPETIDO = "|".join(f"{d}{{7,}}" for d in "0123456789")


def es_relleno(serie_texto):
    """R6: dígito repetido 7+ veces, o la secuencia 123456."""
    digitos = serie_texto.astype(str).str.replace(r"[^\d]", "", regex=True)
    return (digitos.str.fullmatch(PATRON_DIGITO_REPETIDO).fillna(False)
            | digitos.str.contains("123456", na=False))


def sanear_precios(df):
    df["precio_relleno"] = es_relleno(df["precio_texto"])
    df.loc[df["precio_relleno"], ["precio_venta", "precio_arriendo"]] = pd.NA

    for operacion, columna in (("arriendo", "precio_arriendo"), ("venta", "precio_venta")):
        minimo, maximo = RANGO_PRECIO[operacion]
        valores = df[columna]
        fuera = valores.notna() & ((valores < minimo) | (valores > maximo))
        df[f"{columna}_fuera_de_rango"] = fuera
        df.loc[fuera, columna] = pd.NA
        print(f"  {columna:17} fuera de rango: {int(fuera.sum()):4} | válidos: {int(df[columna].notna().sum()):6}")
    return df


print("R6 — precios de relleno:", int(es_relleno(crudo["precio_texto"]).sum()))
print("R5 — rangos por operación:")
crudo = sanear_precios(crudo)

R6 — precios de relleno: 63
R5 — rangos por operación:
  precio_arriendo   fuera de rango:   40 | válidos:  26225
  precio_venta      fuera de rango:   67 | válidos:  24407


## 4. Saneamiento de área — R7 + D4

**R7 — rango único.** Válido en `[20, 2.000]` m². Fuera de ahí es dato mal cargado.

**D4 — `area_posible_lote`.** El hallazgo de negocio del EDA: entre el 79 % y el 92 % de
los outliers bajos de precio/m² son **casas** con área de lote publicada donde debería ir
el área construida. Una casa de 1.000 m² de lote con 150 m² construidos aparece con un
precio/m² ridículamente bajo, y esa fila arrastra hacia abajo cualquier promedio.

El criterio va en **dos pasadas**, porque el área sola no distingue una casa grande de un
lote: se calcula un precio/m² provisional y se marcan las casas cuyo valor cae bajo la
valla inferior de su operación. Es el precio hundido —no el tamaño— lo que delata que el
área mide otra cosa.

**Las vallas se calculan sólo sobre el universo residencial.** Una bodega tiene un
precio/m² estructuralmente bajo y una oficina uno alto; dejarlas dentro del cálculo
corre la valla y hace que D4 marque casas que no debía (o deje pasar las que sí). Por eso
esta sección va **después** de resolver el tipo por slug (sección 1): sin ese dato no se
puede saber qué filas forman el universo.

In [5]:
RANGO_AREA = (20, 2000)


def sanear_areas(df):
    minimo, maximo = RANGO_AREA
    area = pd.to_numeric(df["area_m2"], errors="coerce")

    fuera = area.notna() & ((area < minimo) | (area > maximo))
    df["area_fuera_de_rango"] = fuera
    df["area_m2"] = area.where(~fuera)
    print(f"  R7 — área fuera de [{minimo}, {maximo}]: {int(fuera.sum())} ({fuera.mean() * 100:.2f} %)")

    return df


def precio_unificado(df):
    """El precio que corresponde a la operación de la fila."""
    return df["precio_arriendo"].astype("Float64").fillna(df["precio_venta"].astype("Float64"))


def marcar_area_posible_lote(df):
    """D4, en dos pasadas: el área sola no distingue casa grande de área de lote.

    Lo que la distingue es el precio/m² hundido. Se calcula un precio/m² provisional
    sobre el área declarada y se marcan las casas que caen bajo la valla inferior.

    La valla sale **sólo del universo residencial**: una bodega tiene precio/m²
    estructuralmente bajo y una oficina uno alto, y meterlas en el cálculo mueve el
    umbral que decide sobre las casas.
    """
    provisional = precio_unificado(df) / df["area_m2"]
    es_residencial = ~df["tipo_no_residencial"]

    marca = pd.Series(False, index=df.index)
    for operacion, grupo in df.groupby("operacion"):
        base = grupo.index[es_residencial.loc[grupo.index].to_numpy()]
        inferior, _ = vallas_iqr_log(provisional.loc[base])
        if np.isnan(inferior):
            continue
        # fillna(False): la comparación sobre Float64 nullable devuelve NA donde no hay
        # dato, y un NA no entra en una serie booleana de numpy
        bajo_mercado = (provisional.loc[grupo.index] < inferior).fillna(False).astype(bool)
        es_casa = grupo["tipo_inmueble"].eq("casa").fillna(False).astype(bool)
        marca.loc[grupo.index] = es_casa & bajo_mercado
        print(f"  D4 — {operacion:9} valla inferior de precio/m²: {inferior:>12,.0f}"
              f" | base residencial: {len(base):6,} | casas marcadas: {int((es_casa & bajo_mercado).sum())}")

    df["area_posible_lote"] = marca
    casas = df["tipo_inmueble"].eq("casa")
    print(f"       total: {int(marca.sum())} ({df.loc[casas, 'area_posible_lote'].mean() * 100:.1f} % de las casas)")
    print(f"       área mediana de las marcadas: {df.loc[marca, 'area_m2'].median():,.0f} m²"
          f" | del resto: {df.loc[~marca, 'area_m2'].median():,.0f} m²")
    return df


crudo = sanear_areas(crudo)
crudo = marcar_area_posible_lote(crudo)

  R7 — área fuera de [20, 2000]: 1776 (3.50 %)
  D4 — ambas     valla inferior de precio/m²:    1,460,337 | base residencial:    289 | casas marcadas: 1
  D4 — arriendo  valla inferior de precio/m²:       10,552 | base residencial: 24,248 | casas marcadas: 445
  D4 — venta     valla inferior de precio/m²:    1,632,136 | base residencial: 24,179 | casas marcadas: 429
       total: 875 (4.4 % de las casas)
       área mediana de las marcadas: 1,000 m² | del resto: 126 m²


## 5. Variables censuradas — R8

`habitaciones`, `banos` y `parqueaderos` topan en 5, 5 y 4, con acumulación anómala justo
en el tope. No es la distribución real: es el filtro del sitio agrupando en "5 o más" y
"4 o más".

`habitaciones = 5` significa **`>= 5`**. Se guardan como enteros nullable (`Int8`) y se
agrega una bandera `_es_tope` por variable. Quien modele después decide qué hacer con
ellas — pero con el dato de que están censuradas a la vista, no escondido.

**Nunca imputar con la media**: hacia arriba el valor real es desconocido.

In [6]:
TOPES = {"habitaciones": 5, "banos": 5, "parqueaderos": 4}


def marcar_censuradas(df):
    for columna, tope in TOPES.items():
        valores = pd.to_numeric(df[columna], errors="coerce").astype("Int8")
        df[columna] = valores
        df[f"{columna}_es_tope"] = valores.eq(tope).fillna(False)
        print(f"  {columna:13} tope={tope} | en el tope: {int(df[f'{columna}_es_tope'].sum()):5}"
              f" | nulos: {int(valores.isna().sum()):5}"
              f" | máximo observado: {valores.max()}")
    return df


crudo = marcar_censuradas(crudo)

  habitaciones  tope=5 | en el tope:  5854 | nulos:  2106 | máximo observado: 5
  banos         tope=5 | en el tope:  9740 | nulos:  1075 | máximo observado: 5
  parqueaderos  tope=4 | en el tope:  8342 | nulos: 11342 | máximo observado: 4


## 6. Normalización de texto — R10

`ciudad` y `sector` traen el mismo lugar escrito de varias formas (`CHICO`, `Chico`,
`chico`). Sin normalizar, agrupar por sector produce miles de categorías falsas.

Se hace lo conservador: colapsar espacios, pasar a *title case* y unificar por una clave
sin tildes ni mayúsculas. **No** se hace *fuzzy matching* — juntar `Chico` con
`Chico Norte` sería inventar información; son sectores distintos.

In [7]:
def normalizar_lugar(serie):
    limpio = (serie.astype("string")
              .str.replace(r"\s+", " ", regex=True)
              .str.strip()
              .replace({"": pd.NA}))
    return limpio.str.title()


def normalizar_texto(df):
    for columna in ("ciudad", "sector"):
        antes = df[columna].nunique()
        df[columna] = normalizar_lugar(df[columna])
        # Clave de agrupación insensible a tildes y mayúsculas
        df[f"{columna}_clave"] = df[columna].map(sin_tilde).str.lower()
        print(f"  {columna:8} únicos: {antes:5} -> {df[columna].nunique():5}"
              f" | por clave sin tilde: {df[f'{columna}_clave'].nunique():5}"
              f" | nulos: {int(df[columna].isna().sum())}")
    return df


crudo = normalizar_texto(crudo)
display(crudo["ciudad"].value_counts().head(8).to_frame("anuncios"))

  ciudad   únicos:   271 ->   271 | por clave sin tilde:   271 | nulos: 0


  sector   únicos:  9551 ->  7949 | por clave sin tilde:  7860 | nulos: 510

,anuncios
ciudad,
Bogotá D.C.,15964
Medellín,8790
Envigado,3272
Barranquilla,2659
Cali,2475
Pereira,1579
Chía,1443
Rionegro,1366


## 7. Precio por m² y outliers — D1, D2, D3

Recién ahora se puede calcular `precio_m2`: después de separar los precios (D0), sanear
rangos (R5/R6) y validar el área (R7), y **excluyendo** las casas con área de lote (D4),
donde la razón no es comparable.

Sobre esa base se aplica **IQR de Tukey en escala logarítmica** (**D1**). El EDA mostró
por qué la escala cruda no sirve: la valla inferior da negativa, no detecta un solo
outlier bajo y marca entre el 7 % y el 9 % de las filas. En log baja al 0,6–1,8 % y
detecta las dos colas.

**Segmentación (D2).** Las vallas se calculan por `(operacion, tipo_inmueble)`, porque un
arriendo de apartaestudio y una venta de casa no comparten distribución. Un segmento con
menos de `N_MINIMO_SEGMENTO` observaciones no da una valla confiable y cae al respaldo
global de su operación.

**El respaldo global también es residencial.** Se calcula sobre las filas de vivienda
únicamente: si entraran oficinas y bodegas, un segmento chico se estaría midiendo contra
la dispersión de otro mercado. Las filas no residenciales igual se evalúan y se marcan —
pero no votan sobre dónde va la valla, y de todos modos la compuerta las manda a
cuarentena por `tipo_no_residencial`.

In [8]:
N_MINIMO_SEGMENTO = 300


def derivar_precio_m2(df):
    """Precio/m² definitivo: excluye el área de las casas marcadas como lote (D4)."""
    area_utilizable = df["area_m2"].where(~df["area_posible_lote"])
    df["precio_m2"] = (precio_unificado(df) / area_utilizable).astype("Float64")
    return df


def marcar_outliers(df, columnas=("precio_arriendo", "precio_venta", "area_m2", "precio_m2")):
    """D1-D3. Las vallas describen el mercado residencial; las demás filas se miden
    contra él pero no participan de su cálculo."""
    es_residencial = ~df["tipo_no_residencial"]
    reporte = []
    for columna in columnas:
        marca = pd.Series(False, index=df.index)
        for (operacion, tipo), grupo in df.groupby(["operacion", "tipo_inmueble"], dropna=False):
            serie = grupo[columna]
            usable = pd.to_numeric(serie, errors="coerce").dropna()
            if len(usable) >= N_MINIMO_SEGMENTO:
                inf, sup = vallas_iqr_log(serie)
                origen = "segmento"
            else:
                respaldo = df["operacion"].eq(operacion) & es_residencial
                inf, sup = vallas_iqr_log(df.loc[respaldo, columna])
                origen = "global"
            if np.isnan(inf):
                continue
            valores = pd.to_numeric(serie, errors="coerce")
            fuera = (valores.notna() & ((valores < inf) | (valores > sup))).fillna(False).astype(bool)
            marca.loc[grupo.index] = fuera
            reporte.append({"columna": columna, "operacion": operacion, "tipo": tipo,
                            "n": len(usable), "umbral": origen,
                            "valla_inf": inf, "valla_sup": sup, "outliers": int(fuera.sum())})
        df[f"outlier_{columna}"] = marca
    return df, pd.DataFrame(reporte)


crudo = derivar_precio_m2(crudo)
crudo, reporte_vallas = marcar_outliers(crudo)

print("precio_m2 calculable en", int(crudo["precio_m2"].notna().sum()), "de", len(crudo), "filas")
print("\nVallas del universo residencial (las filas no residenciales van a cuarentena igual):")
display(reporte_vallas[reporte_vallas["tipo"].isin(RESIDENCIAL)].round(0).reset_index(drop=True))

columnas_outlier = [c for c in crudo.columns if c.startswith("outlier_")]
crudo["n_marcas_outlier"] = crudo[columnas_outlier].sum(axis=1)
print("\nFilas con al menos una marca de outlier:",
      int((crudo["n_marcas_outlier"] > 0).sum()),
      f"({(crudo['n_marcas_outlier'] > 0).mean() * 100:.2f} %)")

precio_m2 calculable en 48004 de 50802 filas

Vallas del universo residencial (las filas no residenciales van a cuarentena igual):


,columna,operacion,tipo,n,umbral,valla_inf,valla_sup,outliers
0,precio_arriendo,arriendo,apartaestudio,4199,segmento,557246.0,8.236942e+06,45
1,precio_arriendo,arriendo,apartamento,10216,segmento,689323.0,2.376246e+07,154
2,precio_arriendo,arriendo,casa,9754,segmento,1354299.0,5.685597e+07,197
3,precio_venta,ambas,apartaestudio,2,global,249823898.0,9.956888e+09,0
4,precio_venta,ambas,apartamento,75,global,249823898.0,9.956888e+09,0
5,precio_venta,ambas,casa,210,global,249823898.0,9.956888e+09,2
6,precio_venta,venta,apartaestudio,4236,segmento,103650527.0,1.204046e+09,131
7,precio_venta,venta,apartamento,9794,segmento,72407734.0,6.905340e+09,61
8,precio_venta,venta,casa,10081,segmento,139656581.0,1.235173e+10,44
9,area_m2,ambas,apartaestudio,2,global,58.0,2.131000e+03,2



Filas con al menos una marca de outlier: 1959 (3.86 %)


## 8. Consolidación y compuerta de calidad — R11 + política de cuarentena

**R11** — un solo dataset, con `operacion` como columna. La llave del hecho es
`(id_inmueble, operacion)`. Los duales aparecen en los dos feeds: se consolidan en **una**
fila con `operacion = 'ambas'`, no en dos.

Con el tipo derivado del slug (sección 1) esta consolidación además dejó de ser riesgosa:
antes, las dos copias de un mismo anuncio traían etiquetas de tipo distintas y el
`drop_duplicates` decidía cuál sobrevivía. Ahora las dos traen el mismo tipo — el suyo —
y cuál gane deja de importar.

### Cambio de política respecto de R12

El diseño original marcaba y no borraba, porque con ~20 000 filas descartar salía caro.
Con el barrido por tipo el dataset proyectado ronda las **60 000 filas**, y ahí conviene
lo contrario: que al análisis llegue sólo lo confiable.

Ahora hay **dos salidas**:

| Stage | Contenido |
|---|---|
| `anuncios` | Filas que pasan **todas** las reglas bloqueantes |
| `cuarentena` | El resto, con `motivo_rechazo` explicando por qué |

Nada se pierde: la cuarentena es un archivo, no un `DELETE`. Se audita, y si una regla
resulta demasiado estricta se reprocesa desde ahí.

### No toda bandera bloquea — y acá está el punto fino

Filtrar por *cualquier* bandera sería un error grave. Las tres banderas `_es_tope` marcan
**censura del origen**, no un defecto: el sitio agrupa en "5 o más" y "4 o más". Están en
el **27,4 %** de las filas, y bloquearlas se llevaría justamente el segmento alto del
mercado:

| | pasaría | se iría |
|---|---|---|
| área mediana | 85 m² | **340 m²** |
| precio de venta mediano | $560 M | **$1.850 M** |
| % que son casas | 23,4 % | **83,6 %** |

Eso no elimina sesgo: lo fabrica. Un dato censurado sigue siendo correcto — sólo hay que
saber leerlo. **Pasa, con su bandera puesta.**

Bloquea, en cambio, todo lo que hace al dato **incorrecto o incomparable**:

| Motivo | Por qué bloquea |
|---|---|
| `tipo_no_residencial` | El slug dice oficina, bodega, local o lote: no es el universo del análisis |
| `precio_relleno` | Cifra inventada por el anunciante |
| `precio_*_fuera_de_rango` | Valor imposible para la operación |
| `area_fuera_de_rango` | Área imposible para vivienda |
| `area_posible_lote` | El área mide otra cosa (lote, no construido): no es comparable |
| `titulo_sin_parsear` | No se pudo derivar la estructura del anuncio |
| `outlier_*` | Fuera de la valla IQR log de su segmento |
| `sin_precio` / `sin_area` | Sin las dos variables centrales no hay análisis posible |

In [9]:
# Reglas que mandan la fila a cuarentena: el dato es incorrecto o no comparable
REGLAS_BLOQUEANTES = [
    "tipo_no_residencial",
    "precio_relleno",
    "precio_arriendo_fuera_de_rango",
    "precio_venta_fuera_de_rango",
    "area_fuera_de_rango",
    "area_posible_lote",
    "titulo_sin_parsear",
    "outlier_precio_arriendo",
    "outlier_precio_venta",
    "outlier_area_m2",
    "outlier_precio_m2",
    "sin_precio",
    "sin_area",
]

# Banderas que NO bloquean: describen el dato, no lo invalidan. Las tres `_es_tope`
# son censura del origen y están en el 26 % de las filas — bloquearlas se llevaría
# el segmento alto entero y fabricaría el sesgo que se quiere evitar.
BANDERAS_INFORMATIVAS = [
    "es_dual", "habitaciones_es_tope", "banos_es_tope", "parqueaderos_es_tope",
]


def consolidar(df):
    antes = len(df)
    # Los duales llegan por los dos feeds: una sola fila por (id_inmueble, operacion)
    df = df.sort_values("operacion_feed").drop_duplicates(
        subset=["id_inmueble", "operacion"], keep="first").reset_index(drop=True)
    print(f"  R11 — filas: {antes} -> {len(df)} ({antes - len(df)} consolidadas)")
    return df


def marcar_ausencias(df):
    """Sin precio o sin área no hay análisis posible: son motivos de rechazo por derecho propio."""
    df["sin_precio"] = df["precio_arriendo"].isna() & df["precio_venta"].isna()
    df["sin_area"] = df["area_m2"].isna()
    return df


def separar_por_calidad(df):
    """Compuerta: devuelve (limpio, cuarentena). El motivo viaja con la fila rechazada."""
    for columna in REGLAS_BLOQUEANTES + BANDERAS_INFORMATIVAS:
        df[columna] = df[columna].fillna(False).astype(bool)

    rechaza = df[REGLAS_BLOQUEANTES].any(axis=1)
    df["n_motivos_rechazo"] = df[REGLAS_BLOQUEANTES].sum(axis=1)
    df["motivo_rechazo"] = [
        "|".join(m for m in REGLAS_BLOQUEANTES if fila[m]) or "ok"
        for _, fila in df[REGLAS_BLOQUEANTES].iterrows()
    ]

    limpio = df[~rechaza].reset_index(drop=True)
    cuarentena = df[rechaza].reset_index(drop=True)

    print(f"\n  Compuerta de calidad sobre {len(df):,} filas:")
    print(f"    pasan      : {len(limpio):6,} ({len(limpio) / len(df) * 100:5.2f} %)")
    print(f"    cuarentena : {len(cuarentena):6,} ({len(cuarentena) / len(df) * 100:5.2f} %)")
    return limpio, cuarentena


consolidado = consolidar(crudo)
consolidado = marcar_ausencias(consolidado)
limpio, cuarentena = separar_por_calidad(consolidado)

print("\nMotivos de rechazo (una fila puede tener varios):")
display(cuarentena[REGLAS_BLOQUEANTES].sum().sort_values(ascending=False).to_frame("filas"))

print("Las banderas informativas SIGUEN presentes en el set limpio:")
for bandera in BANDERAS_INFORMATIVAS:
    print(f"  {bandera:22} {int(limpio[bandera].sum()):6} filas ({limpio[bandera].mean() * 100:5.2f} %)")

  R11 — filas: 50802 -> 50761 (41 consolidadas)



  Compuerta de calidad sobre 50,761 filas:
    pasan      : 44,833 (88.32 %)
    cuarentena :  5,928 (11.68 %)

Motivos de rechazo (una fila puede tener varios):


,filas
tipo_no_residencial,2086
area_fuera_de_rango,1776
sin_area,1776
area_posible_lote,875
outlier_area_m2,826
outlier_precio_m2,785
outlier_precio_arriendo,504
outlier_precio_venta,239
sin_precio,169
precio_venta_fuera_de_rango,66


Las banderas informativas SIGUEN presentes en el set limpio:
  es_dual                   238 filas ( 0.53 %)
  habitaciones_es_tope     4967 filas (11.08 %)
  banos_es_tope            7962 filas (17.76 %)
  parqueaderos_es_tope     6515 filas (14.53 %)


## 9. Validación

Antes de dar el flujo por bueno hay que comprobar que hace lo que dice. Estas
verificaciones son las que después van como tests de `transform.py`: si alguna falla, el
pipeline está roto y hay que enterarse acá, no en producción.

In [10]:
CLAVE = ["id_inmueble", "operacion"]


def validar(limpio, cuarentena, consolidado):
    fallas = []

    def revisar(condicion, mensaje):
        estado = "OK  " if condicion else "FALLA"
        print(f"  [{estado}] {mensaje}")
        if not condicion:
            fallas.append(mensaje)

    print("Contrato del stage LIMPIO:")
    revisar(limpio[CLAVE].duplicated().sum() == 0,
            "R2/R11 — la llave (id_inmueble, operacion) es única")
    revisar(limpio["id_inmueble"].notna().all(),
            "R2 — no hay id_inmueble nulo")
    revisar(not limpio["id_inmueble"].str.contains(r"[?&=]", na=False).any(),
            "R2 — el id_inmueble está limpio de querystring")

    # R3 se verifica contra la URL, no contra la columna: si se comparara `tipo_inmueble`
    # consigo mismo la prueba pasaría siempre — que es exactamente cómo se coló la fuga
    # de oficinas y bodegas cuando el tipo venía de la etiqueta del barrido.
    tipo_en_url = tipo_desde_url(limpio["url"])
    revisar(tipo_en_url.isin(RESIDENCIAL).all(),
            f"R3 — el slug de toda URL limpia es uno de {RESIDENCIAL}")
    revisar(limpio["tipo_inmueble"].eq(tipo_en_url).all(),
            "R3 — tipo_inmueble coincide con el slug de su propia URL")
    revisar(not cuarentena.empty
            and not tipo_desde_url(cuarentena.loc[cuarentena["tipo_no_residencial"], "url"])
            .isin(RESIDENCIAL).any(),
            "R3 — nada residencial quedó atrapado en tipo_no_residencial")

    # La compuerta: ninguna regla bloqueante puede sobrevivir en el stage limpio
    revisar(not limpio[REGLAS_BLOQUEANTES].any().any(),
            "Compuerta — ninguna fila limpia dispara una regla bloqueante")
    revisar((limpio["motivo_rechazo"] == "ok").all(),
            "Compuerta — toda fila limpia tiene motivo_rechazo = 'ok'")

    revisar(limpio["area_m2"].notna().all(),
            "Completitud — toda fila limpia tiene área")
    revisar((limpio["precio_arriendo"].notna() | limpio["precio_venta"].notna()).all(),
            "Completitud — toda fila limpia tiene al menos un precio")
    revisar(limpio["precio_m2"].notna().all(),
            "Completitud — toda fila limpia tiene precio_m2 calculable")

    arriendos = limpio.loc[limpio["precio_arriendo"].notna(), "precio_arriendo"]
    revisar(arriendos.between(*RANGO_PRECIO["arriendo"]).all(),
            "R5 — todo precio_arriendo está dentro de rango")
    ventas = limpio.loc[limpio["precio_venta"].notna(), "precio_venta"]
    revisar(ventas.between(*RANGO_PRECIO["venta"]).all(),
            "R5 — todo precio_venta está dentro de rango")
    revisar(not (limpio["es_dual"] & limpio["precio_arriendo"].notna()).any(),
            "D0 — ningún dual conserva canon de arriendo")
    revisar(limpio["area_m2"].between(*RANGO_AREA).all(),
            "R7 — toda área está dentro de rango")

    for columna, tope in TOPES.items():
        revisar(pd.to_numeric(limpio[columna], errors="coerce").max() <= tope,
                f"R8 — {columna} no supera el tope {tope}")

    print("\nLas censuradas NO se filtraron (si se hubieran filtrado, esto falla):")
    censura = ["habitaciones_es_tope", "banos_es_tope", "parqueaderos_es_tope"]
    proporcion = limpio[censura].any(axis=1).mean()
    revisar(proporcion > 0.15,
            f"R8 — el stage limpio conserva las filas censuradas ({proporcion * 100:.1f} %)")

    print("\nContrato del stage CUARENTENA:")
    revisar(cuarentena[REGLAS_BLOQUEANTES].any(axis=1).all(),
            "Toda fila en cuarentena tiene al menos un motivo")
    revisar((cuarentena["motivo_rechazo"] != "ok").all(),
            "Ninguna fila en cuarentena quedó con motivo 'ok'")

    print("\nConservación (nada se pierde, sólo se reparte):")
    revisar(len(limpio) + len(cuarentena) == len(consolidado),
            "La suma de los dos stages es el total consolidado")
    claves_limpio = set(map(tuple, limpio[CLAVE].to_numpy()))
    claves_cuarentena = set(map(tuple, cuarentena[CLAVE].to_numpy()))
    revisar(claves_limpio.isdisjoint(claves_cuarentena),
            "Ninguna clave aparece en los dos stages a la vez")
    revisar(claves_limpio | claves_cuarentena
            == set(map(tuple, consolidado[CLAVE].to_numpy())),
            "La unión de los stages reconstruye el consolidado")

    return fallas


print("Validación del flujo\n")
fallas = validar(limpio, cuarentena, consolidado)
print(f"\n{'TODAS LAS VALIDACIONES PASAN' if not fallas else f'HAY {len(fallas)} FALLAS'}")

Validación del flujo

Contrato del stage LIMPIO:
  [OK  ] R2/R11 — la llave (id_inmueble, operacion) es única
  [OK  ] R2 — no hay id_inmueble nulo
  [OK  ] R2 — el id_inmueble está limpio de querystring
  [OK  ] R3 — el slug de toda URL limpia es uno de ('apartaestudio', 'apartamento', 'casa')
  [OK  ] R3 — tipo_inmueble coincide con el slug de su propia URL
  [OK  ] R3 — nada residencial quedó atrapado en tipo_no_residencial
  [OK  ] Compuerta — ninguna fila limpia dispara una regla bloqueante
  [OK  ] Compuerta — toda fila limpia tiene motivo_rechazo = 'ok'
  [OK  ] Completitud — toda fila limpia tiene área
  [OK  ] Completitud — toda fila limpia tiene al menos un precio
  [OK  ] Completitud — toda fila limpia tiene precio_m2 calculable
  [OK  ] R5 — todo precio_arriendo está dentro de rango
  [OK  ] R5 — todo precio_venta está dentro de rango
  [OK  ] D0 — ningún dual conserva canon de arriendo
  [OK  ] R7 — toda área está dentro de rango
  [OK  ] R8 — habitaciones no supera el top

In [11]:
print("Cobertura de campos en el stage limpio:\n")
display(pd.DataFrame({
    "no nulos": limpio.notna().sum(),
    "% cobertura": (limpio.notna().mean() * 100).round(2),
}).loc[[
    "id_inmueble", "tipo_inmueble", "operacion", "precio_arriendo", "precio_venta",
    "area_m2", "precio_m2", "habitaciones", "banos", "parqueaderos", "ciudad", "sector",
]])

print("¿La compuerta sesgó la composición del dataset?\n")
for columna in ("tipo_inmueble", "operacion"):
    comparacion = pd.DataFrame({
        "consolidado %": consolidado[columna].value_counts(normalize=True) * 100,
        "limpio %": limpio[columna].value_counts(normalize=True) * 100,
        "cuarentena %": cuarentena[columna].value_counts(normalize=True) * 100,
    }).round(1)
    comparacion["desvío"] = (comparacion["limpio %"] - comparacion["consolidado %"]).round(1)
    display(comparacion)

print("\nMedianas del stage limpio:")
display(limpio.groupby(["operacion", "tipo_inmueble"]).agg(
    n=("id_inmueble", "size"),
    area_mediana=("area_m2", "median"),
    precio_m2_mediano=("precio_m2", "median"),
).round(0))

Cobertura de campos en el stage limpio:



,no nulos,% cobertura
id_inmueble,44833,100.00
tipo_inmueble,44833,100.00
operacion,44833,100.00
precio_arriendo,22059,49.20
precio_venta,22774,50.80
area_m2,44833,100.00
precio_m2,44833,100.00
habitaciones,44779,99.88
banos,44799,99.92
parqueaderos,35771,79.79


¿La compuerta sesgó la composición del dataset?



,consolidado %,limpio %,cuarentena %,desvío
tipo_inmueble,,,,
apartaestudio,16.7,17.1,13.9,0.4
apartamento,39.6,43.0,13.7,3.4
bodega,1.1,NaN,9.8,NaN
casa,39.6,39.9,37.2,0.3
consultorio,0.0,NaN,0.2,NaN
edificio,0.2,NaN,1.8,NaN
finca,0.1,NaN,0.7,NaN
local,0.7,NaN,5.6,NaN
lote,0.0,NaN,0.2,NaN


,consolidado %,limpio %,cuarentena %,desvío
operacion,,,,
ambas,0.5,0.5,0.3,0.0
arriendo,51.9,49.2,72.0,-2.7
venta,47.6,50.3,27.7,2.7



Medianas del stage limpio:


n  area_mediana  precio_m2_mediano
operacion tipo_inmueble                                       
ambas     apartamento      56         162.0          5139365.0
          casa            182         450.0          4099206.0
arriendo  apartaestudio  3867          40.0            55556.0
          apartamento    9617          85.0            45455.0
          casa           8575         280.0            31538.0
venta     apartaestudio  3800          40.0          8664474.0
          apartamento    9614          98.0          6875000.0
          casa           9122         274.0          4939492.0

In [12]:
limpio.head()

,id_inmueble,url,texto,tipo_inmueble,tipo_feed,operacion_feed,precio_texto,area_m2,habitaciones,banos,parqueaderos,sector,ciudad,fecha_extraccion,tipo_no_residencial,es_dual,operacion,titulo_sin_parsear,precio_venta,precio_arriendo,precio_relleno,precio_arriendo_fuera_de_rango,precio_venta_fuera_de_rango,area_fuera_de_rango,area_posible_lote,habitaciones_es_tope,banos_es_tope,parqueaderos_es_tope,ciudad_clave,sector_clave,precio_m2,outlier_precio_arriendo,outlier_precio_venta,outlier_area_m2,outlier_precio_m2,n_marcas_outlier,sin_precio,sin_area,n_motivos_rechazo,motivo_rechazo
0,9851-M6839735,https://www.metrocuadrado.com/inmueble/arriend...,Destacado El Poblado | Barranquilla $1.769.280...,apartaestudio,apartaestudio,arriendo,$1.769.280,39.0,1,1,1,El Poblado,Barranquilla,2026-09-10T04:25:40+00:00,False,False,arriendo,False,<NA>,1769280,False,False,False,False,False,False,False,False,barranquilla,el poblado,45366.153846,False,False,False,False,0,False,False,0,ok
1,22335-M6523328,https://www.metrocuadrado.com/inmueble/arriend...,Destacado CIUDAD DEL RIO | Suroriente | Medell...,apartaestudio,apartaestudio,arriendo,$2.450.777,32.0,1,1,<NA>,Ciudad Del Rio,Medellín,2026-09-10T04:25:40+00:00,False,False,arriendo,False,<NA>,2450777,False,False,False,False,False,False,False,False,medellin,ciudad del rio,76586.78125,False,False,False,False,0,False,False,0,ok
2,21003-M6817169,https://www.metrocuadrado.com/inmueble/arriend...,Destacado Los Monjes | Bogotá D.C. $1.600.000 ...,apartaestudio,apartaestudio,arriendo,$1.600.000,29.0,1,1,<NA>,Los Monjes,Bogotá D.C.,2026-09-10T04:25:40+00:00,False,False,arriendo,False,<NA>,1600000,False,False,False,False,False,False,False,False,bogota d.c.,los monjes,55172.413793,False,False,False,False,0,False,False,0,ok
3,MC6913630,https://www.metrocuadrado.com/inmueble/arriend...,Modelia | Occidental | Bogotá D.C. $1.700.000 ...,apartaestudio,apartaestudio,arriendo,$1.700.000,37.0,1,1,<NA>,Modelia,Bogotá D.C.,2026-09-10T04:25:40+00:00,False,False,arriendo,False,<NA>,1700000,False,False,False,False,False,False,False,False,bogota d.c.,modelia,45945.945946,False,False,False,False,0,False,False,0,ok
4,23443-M7034689,https://www.metrocuadrado.com/inmueble/arriend...,Destacado PARKWAY | Chapinero | Bogotá D.C. Ba...,apartaestudio,apartaestudio,arriendo,$2.100.000,35.0,1,1,<NA>,Parkway,Bogotá D.C.,2026-09-10T04:25:40+00:00,False,False,arriendo,False,<NA>,2100000,False,False,False,False,False,False,False,False,bogota d.c.,parkway,60000.0,False,False,False,False,0,False,False,0,ok


## 10. Salida — dos stages

```
data/processed/
├── anuncios.{csv,parquet}     ← stage limpio: lo que consume el análisis
├── cuarentena.{csv,parquet}   ← stage de riesgo: rechazadas + motivo_rechazo
└── transform_resumen.json     ← conteos de la corrida, para monitoreo
```

**Parquet además de CSV** porque preserva los tipos: los `Int8` nullable, los booleanos y
las fechas que el CSV degrada a texto y obliga a re-inferir en cada lectura.

El resumen JSON es el que hay que mirar en cada corrida periódica: si la tasa de rechazo
se mueve de golpe, o el sitio cambió o una regla dejó de servir.

In [13]:
def escribir(df, directorio, nombre):
    directorio.mkdir(parents=True, exist_ok=True)
    df.to_csv(directorio / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    df.to_parquet(directorio / f"{nombre}.parquet", index=False)
    print(f"  {nombre:12} {len(df):6,} filas x {len(df.columns):2} columnas")


def exportar(limpio, cuarentena, directorio=DIR_PROCESSED):
    print("Escribiendo stages:")
    escribir(limpio, directorio, "anuncios")
    escribir(cuarentena, directorio, "cuarentena")

    total = len(limpio) + len(cuarentena)
    todo = pd.concat([limpio, cuarentena], ignore_index=True)
    resumen = {
        "total_procesado": total,
        "limpio": len(limpio),
        "cuarentena": len(cuarentena),
        "tasa_rechazo_%": round(len(cuarentena) / total * 100, 2),
        "motivos_rechazo": {m: int(cuarentena[m].sum())
                            for m in REGLAS_BLOQUEANTES if cuarentena[m].sum() > 0},
        "limpio_por_operacion": limpio["operacion"].value_counts().to_dict(),
        "limpio_por_tipo": limpio["tipo_inmueble"].value_counts().to_dict(),
        "limpio_con_censura_%": round(
            limpio[["habitaciones_es_tope", "banos_es_tope", "parqueaderos_es_tope"]]
            .any(axis=1).mean() * 100, 2),
        # Salud de la fuente: cuánto discrepa el barrido del slug. Si esto se dispara,
        # el filtro del sitio cambió y hay que revisar el extract.
        "reetiquetadas_por_slug": int((~todo["tipo_inmueble"].eq(todo["tipo_feed"])).sum()),
        "acuerdo_slug_barrido_%": round(
            todo["tipo_inmueble"].eq(todo["tipo_feed"]).mean() * 100, 2),
    }
    ruta = directorio / "transform_resumen.json"
    ruta.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  {ruta.name}")
    return resumen


resumen = exportar(limpio, cuarentena)
print()
print(json.dumps(resumen, ensure_ascii=False, indent=2))

Escribiendo stages:


  anuncios     44,833 filas x 40 columnas
  cuarentena    5,928 filas x 40 columnas
  transform_resumen.json

{
  "total_procesado": 50761,
  "limpio": 44833,
  "cuarentena": 5928,
  "tasa_rechazo_%": 11.68,
  "motivos_rechazo": {
    "tipo_no_residencial": 2086,
    "precio_relleno": 63,
    "precio_arriendo_fuera_de_rango": 40,
    "precio_venta_fuera_de_rango": 66,
    "area_fuera_de_rango": 1776,
    "area_posible_lote": 875,
    "outlier_precio_arriendo": 504,
    "outlier_precio_venta": 239,
    "outlier_area_m2": 826,
    "outlier_precio_m2": 785,
    "sin_precio": 169,
    "sin_area": 1776
  },
  "limpio_por_operacion": {
    "venta": 22536,
    "arriendo": 22059,
    "ambas": 238
  },
  "limpio_por_tipo": {
    "apartamento": 19287,
    "casa": 17879,
    "apartaestudio": 7667
  },
  "limpio_con_censura_%": 27.35,
  "reetiquetadas_por_slug": 5495,
  "acuerdo_slug_barrido_%": 89.17
}


## Estado del boceto

El flujo corre completo y está escrito para los dos esquemas: `cargar()` detecta cuál es
y todo lo demás trabaja sobre el contrato normalizado.

### Lo que hay que revisar antes de pasarlo a `src/transform.py`

1. **`area_posible_lote` bloquea, y es la regla más discutible.** Rechaza el 6 % de las
   casas. El argumento a favor: el área mide otra cosa (lote, no construido) y mezclar dos
   unidades en la misma columna arruina cualquier análisis por m². El argumento en contra:
   son inmuebles legítimos y sacarlos sesga contra las casas con lote grande. Es la única
   regla bloqueante que quita datos **correctos** — moverla a informativa es cambiar una
   línea.

2. **Los rangos de R5 y R7 son criterio de negocio, no estadística.** Salieron del EDA y
   son razonables, pero conviene que los valide alguien que conozca el mercado.

3. **Las vallas de outliers y el umbral de D4 se recalculan en cada corrida.** Es
   adaptativo, lo cual es bueno mientras el dataset crece, pero significa que dos corridas
   pueden clasificar distinto la misma fila. Con ejecución periódica hay que decidir si se
   congelan contra una línea base.

4. **Vigilar dos números del resumen.** La `tasa_rechazo_%` y el nuevo
   `acuerdo_slug_barrido_%`. Si la primera salta de golpe, o el sitio cambió o el parseo se
   rompió. Si el segundo baja, el filtro de la fuente está devolviendo cada vez más basura
   y el problema está en el extract, no acá. Son las dos alarmas más baratas del pipeline.

### Deuda que queda en el extract

El transform ya no depende de la etiqueta del barrido, así que la fuga está contenida —
pero sigue existiendo aguas arriba. `extract.py` desperdicia horas de scraping trayendo
oficinas y bodegas, y su `drop_duplicates(keep="first")` sigue asignando el tipo por orden
de barrido. Arreglarlo allá es eficiencia, no corrección: el dato que llega al análisis ya
está bien.

### Lo que falta para producción

- Portar a `src/transform.py` con la misma estructura de funciones (ya están separadas
  para que el traslado sea directo).
- Convertir la sección 9 en tests de verdad.
- Encadenar `extract.py` → `transform.py` en un único punto de entrada.

In [14]:
print("Stage LIMPIO — muestra:")
display(limpio[["id_inmueble", "tipo_inmueble", "operacion", "precio_arriendo",
                "precio_venta", "area_m2", "precio_m2", "habitaciones",
                "habitaciones_es_tope", "ciudad"]].head())

print("\nStage CUARENTENA — muestra, con el motivo:")
display(cuarentena[["id_inmueble", "tipo_inmueble", "operacion", "precio_arriendo",
                    "precio_venta", "area_m2", "motivo_rechazo"]].head(10))

Stage LIMPIO — muestra:


,id_inmueble,tipo_inmueble,operacion,precio_arriendo,precio_venta,area_m2,precio_m2,habitaciones,habitaciones_es_tope,ciudad
0,9851-M6839735,apartaestudio,arriendo,1769280,<NA>,39.0,45366.153846,1,False,Barranquilla
1,22335-M6523328,apartaestudio,arriendo,2450777,<NA>,32.0,76586.78125,1,False,Medellín
2,21003-M6817169,apartaestudio,arriendo,1600000,<NA>,29.0,55172.413793,1,False,Bogotá D.C.
3,MC6913630,apartaestudio,arriendo,1700000,<NA>,37.0,45945.945946,1,False,Bogotá D.C.
4,23443-M7034689,apartaestudio,arriendo,2100000,<NA>,35.0,60000.0,1,False,Bogotá D.C.



Stage CUARENTENA — muestra, con el motivo:


,id_inmueble,tipo_inmueble,operacion,precio_arriendo,precio_venta,area_m2,motivo_rechazo
0,MC6980340,apartaestudio,arriendo,1800000,<NA>,NaN,area_fuera_de_rango|sin_area
1,72-M6664683,apartaestudio,arriendo,2850000,<NA>,NaN,area_fuera_de_rango|sin_area
2,17285-M7017151,apartaestudio,arriendo,8600000,<NA>,165.0,outlier_precio_arriendo|outlier_area_m2
3,23054-M6942352,apartaestudio,arriendo,8900000,<NA>,128.0,outlier_precio_arriendo|outlier_area_m2
4,11589-M6930031,apartaestudio,arriendo,1900000,<NA>,NaN,area_fuera_de_rango|sin_area
5,14774-M6917874,apartaestudio,arriendo,1200000,<NA>,NaN,area_fuera_de_rango|sin_area
6,20837-M7000782,apartaestudio,arriendo,2100000,<NA>,NaN,area_fuera_de_rango|sin_area
7,MC5724210,apartaestudio,arriendo,850000,<NA>,NaN,area_fuera_de_rango|sin_area
8,17556-M6697918,apartaestudio,arriendo,7900000,<NA>,132.0,outlier_area_m2
9,MC2845286,apartaestudio,arriendo,940000,<NA>,NaN,area_fuera_de_rango|sin_area
